In [ ]:
# 06 – Bowling Role Analysis

Purpose:
- Identify bowling specialists by phase
- Classify bowlers into Powerplay, Middle Overs, and Death roles

Input:
- data/processed/bowling_metrics.csv

Outputs:
- data/processed/first_bowlers.csv
- data/processed/middle_bowlers.csv
- data/processed/death_bowlers.csv

In [1]:
# =========================================================
# 06 – Bowling Role Analysis (STRICT ROLE FILTERS)
# =========================================================

import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

from config import *
import pandas as pd


# ---------------------------------------------------------
# Load Data
# ---------------------------------------------------------

df = pd.read_csv(PROCESSED_DIR / "deliveries_cleaned.csv")
bowling = pd.read_csv(PROCESSED_DIR / "bowling_metrics.csv")

print("Deliveries:", df.shape)
print("Bowling metrics:", bowling.shape)


# ---------------------------------------------------------
# Phase Balls Bowled
# ---------------------------------------------------------

phase_balls = (
    df.groupby(["bowler","phase"])
      .size()
      .unstack(fill_value=0)
      .reset_index()
)

phase_balls = phase_balls.rename(columns={
    "Powerplay":"powerplay_balls_bowled",
    "Middle":"middle_overs_balls_bowled",
    "Death":"death_overs_balls_bowled"
})


# ---------------------------------------------------------
# Phase Runs Conceded
# ---------------------------------------------------------

phase_runs = (
    df.groupby(["bowler","phase"])["runs_total"]
      .sum()
      .unstack(fill_value=0)
      .reset_index()
)

phase_runs = phase_runs.rename(columns={
    "Powerplay":"powerplay_runs",
    "Middle":"middle_runs",
    "Death":"death_runs"
})


# ---------------------------------------------------------
# Merge Phase Data
# ---------------------------------------------------------

bowling = bowling.merge(phase_balls, on="bowler", how="left")
bowling = bowling.merge(phase_runs, on="bowler", how="left")


# ---------------------------------------------------------
# Total Balls Bowled
# ---------------------------------------------------------

bowling["total_balls_bowled"] = (
    bowling["powerplay_balls_bowled"]
    + bowling["middle_overs_balls_bowled"]
    + bowling["death_overs_balls_bowled"]
)


# ---------------------------------------------------------
# Economy by Phase
# ---------------------------------------------------------

bowling["powerplay_economy"] = (
    bowling["powerplay_runs"] /
    (bowling["powerplay_balls_bowled"] / 6)
)

bowling["middle_overs_economy"] = (
    bowling["middle_runs"] /
    (bowling["middle_overs_balls_bowled"] / 6)
)

bowling["death_overs_economy"] = (
    bowling["death_runs"] /
    (bowling["death_overs_balls_bowled"] / 6)
)


# Replace infinite values
bowling = bowling.replace([float("inf"), -float("inf")], 0)


# =========================================================
# ROLE 1 — POWERPLAY BOWLERS
# =========================================================

powerplay_bowlers = bowling[
    (bowling["powerplay_balls_bowled"] >= 500) &
    (bowling["powerplay_balls_bowled"] >= 0.30 * bowling["total_balls_bowled"]) &
    (bowling["powerplay_balls_bowled"] > bowling["middle_overs_balls_bowled"]) &
    (bowling["powerplay_balls_bowled"] > bowling["death_overs_balls_bowled"]) &
    (bowling["powerplay_economy"] <= 8.0) &
    (bowling["economy"] <= 8.5) &
    (bowling["wickets"] >= 100)
].copy()

powerplay_bowlers = powerplay_bowlers.sort_values(
    "powerplay_balls_bowled",
    ascending=False
)


# =========================================================
# ROLE 2 — MIDDLE OVERS BOWLERS
# =========================================================

middle_bowlers = bowling[
    (bowling["middle_overs_balls_bowled"] >= 800) &
    (bowling["middle_overs_balls_bowled"] >= 0.40 * bowling["total_balls_bowled"]) &
    (bowling["middle_overs_balls_bowled"] > bowling["powerplay_balls_bowled"]) &
    (bowling["middle_overs_balls_bowled"] > bowling["death_overs_balls_bowled"]) &
    (bowling["middle_overs_economy"] <= 7.5) &
    (bowling["economy"] <= 8.0) &
    (bowling["wickets"] >= 135)
].copy()

middle_bowlers = middle_bowlers.sort_values(
    "middle_overs_balls_bowled",
    ascending=False
)


# =========================================================
# ROLE 3 — DEATH OVERS BOWLERS
# =========================================================

death_bowlers = bowling[
    (bowling["death_overs_balls_bowled"] >= 500) &
    (bowling["death_overs_balls_bowled"] >= 0.30 * bowling["total_balls_bowled"]) &
    (bowling["death_overs_balls_bowled"] > bowling["powerplay_balls_bowled"]) &
    (bowling["death_overs_economy"] <= 10.5) &
    (bowling["economy"] <= 9.0) &
    (bowling["wickets"] >= 100)
].copy()

death_bowlers = death_bowlers.sort_values(
    "death_overs_balls_bowled",
    ascending=False
)


# =========================================================
# Save Role Datasets
# =========================================================

powerplay_bowlers.to_csv(
    PROCESSED_DIR / "first_bowlers.csv",
    index=False
)

middle_bowlers.to_csv(
    PROCESSED_DIR / "middle_bowlers.csv",
    index=False
)

death_bowlers.to_csv(
    PROCESSED_DIR / "death_bowlers.csv",
    index=False
)


# =========================================================
# Quick Check
# =========================================================

print("\nPowerplay bowlers:", powerplay_bowlers.shape)
print("Middle overs bowlers:", middle_bowlers.shape)
print("Death bowlers:", death_bowlers.shape)

powerplay_bowlers.head()

Deliveries: (278205, 15)
Bowling metrics: (551, 12)

Powerplay bowlers: (8, 22)
Middle overs bowlers: (6, 22)
Death bowlers: (4, 22)


,bowler,runs_conceded,balls,wickets,overs,economy,Death_x,Middle_x,Powerplay_x,Death_y,...,death_overs_balls_bowled,middle_overs_balls_bowled,powerplay_balls_bowled,death_runs,middle_runs,powerplay_runs,total_balls_bowled,powerplay_economy,middle_overs_economy,death_overs_economy
72,B Kumar,5336,4222,198,703.666667,7.583136,9.186858,8.374408,6.438649,1461,...,1535,432,2411,2326,601,2614,4378,6.505185,8.347222,9.091857
476,Sandeep Sharma,3990,3045,146,507.500000,7.862069,9.956931,7.901961,6.926627,743,...,790,638,1740,1294,833,2007,3168,6.920690,7.833856,9.827848
499,TA Boult,3678,2687,143,447.833333,8.212877,10.443828,8.627119,7.124069,721,...,752,371,1662,1288,538,1980,2785,7.148014,8.700809,10.276596
313,Mohammed Shami,3688,2615,133,435.833333,8.461950,10.506667,8.337662,7.566982,675,...,700,475,1533,1214,663,1957,2708,7.659491,8.374737,10.405714
510,UT Yadav,4262,3057,144,509.500000,8.365064,10.155738,8.075601,7.636364,732,...,777,901,1512,1312,1204,1926,3190,7.642857,8.017758,10.131274
